# Bangla Health Paraphrase — Full End-to-End Pipeline

This notebook runs the **complete research workflow** in one place. Each section is checkpoint-aware: re-running the notebook skips stages that already finished.

## Research Goal
Build and evaluate a Bangla healthcare paraphrase generator using:
- **PySpark** for distributed preprocessing, EDA, and augmentation
- **mT5-small + LoRA** as the main model
- **Baseline**: mt5-small full fine-tuning (same backbone as LoRA model)

## Hypotheses Tested
| ID | Hypothesis | Section |
|----|-----------|---------|
| H1 | mT5 + LoRA outperforms full-FT baselines | §8–§10 Evaluation |
| H2 | Domain preprocessing improves quality | Compare cleaned vs uncleaned runs |
| H3 | Augmentation improves robustness | Toggle `training.use_augmented_train` |

## Pipeline Overview
```
CSV → Spark preprocess → EDA → Augment → Tokenize → Train (2 models) → Evaluate → Plots → Final report
```

**Tip:** Set `dataset.dev_subset: 10000` in `configs/experiment_config.yaml` for fast smoke tests on a GTX 1070.

---
## §1 Environment Setup

Add the project root to `sys.path`, load the experiment config, create output directories, and fix random seeds for reproducibility.

**Outputs:** `outputs/{checkpoints,logs,metrics,figures,reports,mlruns,.stages}`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.common.config import load_config
from src.common.paths import ensure_dirs, PROJECT_ROOT as ROOT
from src.common.seed import set_seed
from src.common.logging import get_logger

CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment_config.yaml"
cfg = load_config(CONFIG_PATH)
ensure_dirs()
set_seed(cfg.experiment.seed)
logger = get_logger("full_pipeline")

print(f"Project root : {PROJECT_ROOT}")
print(f"Experiment   : {cfg.experiment.name}")
print(f"Seed         : {cfg.experiment.seed}")
print(f"Dev subset   : {cfg.dataset.dev_subset or 'full dataset'}")

---
## §2 Configuration Review

Inspect key settings before running the pipeline. Edit `configs/experiment_config.yaml` to change splits, batch sizes, augmentation, or dev subset.

In [ ]:
print("=== Dataset ===")
print(f"  Source CSV  : {cfg.dataset.raw_csv}")
print(f"  Splits      : train={cfg.dataset.train_split}, val={cfg.dataset.validation_split}, test={cfg.dataset.test_split}")
print(f"  Max lengths : input={cfg.dataset.max_input_length}, target={cfg.dataset.max_target_length}")

print("\n=== Models ===")
for spec in cfg.all_model_specs():
    lora_tag = "+ LoRA" if spec.use_lora else "full FT"
    print(f"  {spec.key:16s} {spec.name} ({lora_tag}, batch={spec.train_batch_size}, accum={spec.gradient_accumulation_steps})")

print("\n=== Training ===")
print(f"  Epochs      : {cfg.training.epochs}")
print(f"  LR          : {cfg.training.learning_rate}")
print(f"  FP16        : {cfg.training.fp16}")
print(f"  Augmented   : {cfg.training.use_augmented_train}")

print("\n=== Evaluation metrics ===")
print(f"  {', '.join(cfg.evaluation.metrics)}")
print(f"  BERTScore   : {cfg.evaluation.bertscore_model}")
print(f"  Semantic    : {cfg.evaluation.semantic_models}")

---
## §3 Spark Preprocessing

Distributed data pipeline (PySpark):
1. **Ingest** — read `datasets/all_paraphrased_data.csv` (UTF-8, Bangla schema)
2. **Profile (raw)** — record nulls, duplicate IDs, Bangla coverage (no strict gate)
3. **Clean** — whitespace trim, drop null/trivial/short/long pairs
4. **Deduplicate** — exact hash dedup + optional MinHashLSH near-duplicate pruning
5. **Validate (clean)** — strict check: zero null sentences after cleaning
6. **Feature engineering** — length, overlap, Bangla char ratio
7. **Split** — 80 / 10 / 10 train / val / test (seed=42)

**Outputs:**
- `data/raw/raw.parquet`
- `data/interim/featured.parquet`
- `data/processed/{train,val,test}.parquet`
- `outputs/reports/raw_validation_summary.json`
- `outputs/reports/validation_summary.json`

_Re-run the preprocess cell after code changes (it reloads `src` modules). Prefer **Kernel → Restart** if imports still look stale._

In [ ]:
from src.common.checkpointing import is_done, mark_done, stage_path
from src.common.reload import assert_preprocess_pipeline, reload_preprocess_modules

reload_preprocess_modules()
from src.pipeline.preprocess import run_preprocess

assert_preprocess_pipeline()
print("[OK] Pipeline order: profile_raw → clean → deduplicate → validate")

from src.common.paths import DATA_DIR, parquet_split_ready

STAGE = "preprocess"
split_paths = [DATA_DIR / "processed" / f"{s}.parquet" for s in ("train", "val", "test")]
if is_done(STAGE, cfg) and all(parquet_split_ready(p) for p in split_paths):
    print(f"[SKIP] {STAGE} already complete → {stage_path(STAGE, cfg)}")
else:
    missing = [p.name for p in split_paths if not p.exists()]
    if is_done(STAGE, cfg) and missing:
        print(f"[WARN] Sentinel exists but missing splits: {missing}; re-running preprocess …")
    else:
        print("[RUN]  Starting Spark preprocessing …")
    run_preprocess(cfg)
    print(f"[DONE] Preprocessing finished. Sentinel: {stage_path(STAGE, cfg)}")

---
## §4 Exploratory Data Analysis (EDA)

Compute dataset statistics entirely in Spark:
- Sentence length distributions (source & target)
- Vocabulary size and top-50 tokens
- Length ratio and word-overlap distributions
- Missing-value summary

**Outputs:**
- `outputs/reports/eda_report.html` — interactive HTML report
- `outputs/reports/eda_stats.json` — raw statistics
- `outputs/figures/eda/*.png` and `*.pdf` (300 DPI)

In [ ]:
from src.reports.eda_report import render_eda_report

STAGE = "eda_report"
if is_done(STAGE, cfg):
    print(f"[SKIP] {STAGE} already complete")
else:
    print(f"[RUN]  Generating EDA report …")

eda_path = render_eda_report(cfg)
print(f"[DONE] EDA report → {eda_path}")

In [ ]:
# Preview EDA stats inline
import json

stats_path = PROJECT_ROOT / cfg.outputs.reports / "eda_stats.json"
if stats_path.exists():
    stats = json.loads(stats_path.read_text(encoding="utf-8"))
    print(f"Total rows        : {stats['total_rows']:,}")
    print(f"Source len (mean) : {stats['src_len_mean']:.1f} tokens")
    print(f"Target len (mean) : {stats['tgt_len_mean']:.1f} tokens")
    print(f"Vocabulary size   : {stats['vocab_size']:,}")
    print(f"Word overlap mean : {stats['word_overlap_mean']:.3f}")
else:
    print("EDA stats not found — run §4 first.")

---
## §5 Data Augmentation (H3)

Apply **token-level EDA augmentation** on the training split via Spark UDFs:
- Synonym replacement (Bangla wordnet via `bnlp_toolkit`)
- Random adjacent token swap
- Random token deletion (preserving minimum length)

Augmented rows are concatenated with originals. Toggle H3 ablation with `training.use_augmented_train` in config.

**Outputs:** `data/augmented/train.parquet`

In [ ]:
from src.spark.augmentation import augment_train

STAGE = "augmentation"
if is_done(STAGE, cfg):
    print(f"[SKIP] {STAGE} already complete")
else:
    print(f"[RUN]  Running augmentation (synonym={cfg.augmentation.synonym_prob}, swap={cfg.augmentation.swap_prob}, del={cfg.augmentation.deletion_prob}) …")

aug_path = augment_train(cfg)
print(f"[DONE] Augmented train set → {aug_path}")
print(f"       use_augmented_train = {cfg.training.use_augmented_train}")

---
## §6 Tokenization

Tokenize both models and all three splits. Each (model, split) pair is cached to disk and skipped on re-run.

| Model | Prefix |
|-------|--------|
| mT5 (main + baseline) | `"paraphrase: "` prefix |

**Outputs:** `data/processed/tokenized/<model_key>/{train,val,test}/`

In [ ]:
from src.data.tokenize import tokenize_all

print("[RUN]  Tokenizing all models and splits …")
tokenize_all(config=cfg)
print("[DONE] Tokenization complete for:", [s.key for s in cfg.all_model_specs()])

---
## §7 Train mT5-small + LoRA (Main Model)

Primary model for **H1**. PEFT LoRA attached to `google/mt5-small`:
- `r=16, alpha=32, dropout=0.1`, target modules `[q, v]`
- Seq2SeqTrainer with cosine scheduler, label smoothing, early stopping
- FP16 + gradient checkpointing (GTX 1070)
- MLflow tracking → `outputs/mlruns/`

**Outputs:** `outputs/checkpoints/mt5_lora/final/`

In [ ]:
from src.models.mt5_lora import run_mt5_lora

STAGE = "train_mt5_lora"
if is_done(STAGE, cfg):
    print(f"[SKIP] {STAGE} already complete")
else:
    print(f"[RUN]  Training mT5-small + LoRA ({cfg.training.epochs} epochs) …")
    ckpt = run_mt5_lora(cfg)
    print(f"[DONE] Checkpoint → {ckpt / 'final'}")

---
## §8 Train Baselines

### mt5-small Full Fine-Tuning
Same architecture as main model but **without LoRA** (all parameters updated). This is the H1 comparison baseline.

**Output:** `outputs/checkpoints/mt5_baseline/final/`

In [ ]:
from src.models.mt5_baseline import run_mt5_baseline

STAGE = "train_mt5_baseline"
if is_done(STAGE, cfg):
    print(f"[SKIP] {STAGE} already complete")
else:
    print(f"[RUN]  Training mt5-small baseline (full FT) …")
    ckpt = run_mt5_baseline(cfg)
    print(f"[DONE] Checkpoint → {ckpt / 'final'}")

---
## §9 Evaluation on Test Set

Generate paraphrases on the **held-out test split** and compute:

| Metric | Tool |
|--------|------|
| BLEU | sacrebleu |
| ROUGE-L | rouge-score |
| BERTScore | xlm-roberta-large |
| Distinct-1 / Distinct-2 | n-gram uniqueness |
| SemSim (mpnet) | paraphrase-multilingual-mpnet-base-v2 |
| SemSim (bn-sbert) | bengali-sentence-similarity-sbert |

Generation: `num_beams=4`, `no_repeat_ngram_size=3`, `max_new_tokens=128`

**Outputs:**
- `outputs/metrics/<model_key>.json`
- `outputs/metrics/summary.csv`

In [ ]:
from src.evaluation.evaluator import evaluate_all, evaluate_model

print("[RUN]  Evaluating all models on test set …")
results = evaluate_all(cfg, force=True)

print("\n=== Test Set Results ===")
for r in results:
    print(f"\n  {r['model_key']} ({r['model_name']})")
    print(f"    BLEU       : {r['BLEU']:.2f}")
    print(f"    ROUGE-L    : {r['ROUGE-L']:.4f}")
    print(f"    BERTScore  : {r['BERTScore']:.4f}")
    print(f"    Distinct-1 : {r['Distinct-1']:.4f}")
    print(f"    Distinct-2 : {r['Distinct-2']:.4f}")
    if "semsim_mpnet" in r:
        print(f"    SemSim mpnet: {r['semsim_mpnet']:.4f}")
    if "semsim_bnsbert" in r:
        print(f"    SemSim bn   : {r['semsim_bnsbert']:.4f}")

---
## §10 Results Visualization

Generate publication-ready figures (300 DPI PNG + PDF) and export result tables:
- Combined multi-panel bar chart (`all_metrics_comparison`) plus per-metric plots
- Training / validation loss curves from MLflow
- CSV summary + LaTeX table for the paper

**Outputs:**
- `outputs/figures/comparison/all_metrics_comparison.png` (all metrics in one figure)
- `outputs/figures/comparison/*.png` (optional per-metric plots)
- `outputs/figures/training/*.png`
- `outputs/reports/results_table.tex`

In [ ]:
from src.visualization.results_plots import generate_all_plots
from src.visualization.tables import export_results

print("[RUN]  Generating plots and tables …")
generate_all_plots()
csv_path, tex_path = export_results(cfg)
print(f"[DONE] Summary CSV  → {csv_path}")
print(f"[DONE] LaTeX table  → {tex_path}")

In [ ]:
# Display summary table inline
import pandas as pd

summary_path = PROJECT_ROOT / cfg.outputs.metrics / "summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print("Summary CSV not found — run §9 first.")

---
## §11 Final Research Report

Compile all artifacts into a single HTML report linking EDA, metrics, plots, and the config snapshot.

In [ ]:
from src.reports.final_report import render_final_report

report_path = render_final_report()
print(f"[DONE] Final report → {report_path}")

---
## §12 Pipeline Status & Artifact Checklist

Review which stages completed and where outputs live.

In [ ]:
STAGES = [
    "preprocess",
    "eda_report",
    "augmentation",
    "train_mt5_lora",
    "train_mt5_baseline",
    "evaluate_mt5_lora",
    "evaluate_mt5_baseline",
]

print(f"{'Stage':<28s} {'Status':<8s}")
print("-" * 38)
for stage in STAGES:
    status = "DONE" if is_done(stage, cfg) else "pending"
    print(f"{stage:<28s} {status}")

ARTIFACTS = [
    ("Processed train",  "data/processed/train.parquet"),
    ("EDA report",       "outputs/reports/eda_report.html"),
    ("Augmented train",  "data/augmented/train.parquet"),
    ("Metrics summary",  "outputs/metrics/summary.csv"),
    ("Final report",     "outputs/reports/final_report.html"),
    ("LaTeX table",      "outputs/reports/results_table.tex"),
]

print("\n=== Artifact Checklist ===")
for label, rel in ARTIFACTS:
    path = PROJECT_ROOT / rel
    mark = "OK" if path.exists() else "--"
    print(f"  [{mark}] {label:<20s} {rel}")

---
## §13 Optional: Force Re-run a Stage

Uncomment and run the cell below to **clear a specific checkpoint** and re-run that stage on the next execution.

In [ ]:
# from src.common.checkpointing import clear_stage

# Example: force re-preprocessing
# clear_stage("preprocess", cfg)
# clear_stage("eda_report", cfg)
# clear_stage("augmentation", cfg)

# Example: force re-train main model
# clear_stage("train_mt5_lora", cfg)
# clear_stage("evaluate_mt5_lora", cfg)

print("Uncomment lines above to clear specific stage sentinels.")

---
## §14 Cleanup

Stop the Spark session to free JVM memory when the notebook is finished.

In [ ]:
from src.spark.session import stop_spark

stop_spark()
print("Spark session stopped. Pipeline complete.")